<a href="https://colab.research.google.com/github/abdulwahab-git/week-01-Assignment/blob/main/work/notebook/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwahab-git/week-01-Assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring.**

My lane is **content refresh prioritization**: the decision is *which content items should an editor review first?* The output is a priority score used to rank pages for a limited editorial review queue.

This is a ranking/scoring problem rather than just classification because the action is to order many pages and take the highest-priority items first. The ML output is decision support for an editor, not a claim about Google's algorithm.

The main decision is: **which pages should the content team spend its limited refresh time on first?** A wrong call can waste editor time or cause a genuinely declining page to be missed.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


In [3]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Starter dataset not found at {DATA_PATH}. "
        "Run this notebook from the root of your FlyRank repo, where the starter CSV lives."
    )

df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print("Lane: content refresh prioritization")


Rows: 30,000
Columns: 44
Lane: content refresh prioritization


## 2. Target or proxy

The target I would use for the real ML system is an **observed future decline outcome** measured in a later time window: for example, whether the content item declines during the next evaluation window after the features are observed.

This matters because the framing skill says the target should be observed rather than created from the same rule the model is supposed to learn. The starter dataset contains `is_declining_label`, but that label is derived from `trend_direction`, which is itself derived from `trend_pct`; therefore I will **not** treat those columns as legitimate predictive features.

For this Week-2 framing exercise, the starter slice is used to show the **unit of analysis and available inputs**. I sketch the future target as `future_decline_target` rather than pretending that the current snapshot contains a clean future outcome.

In [4]:
# The starter label is useful for inspection, but it is not a valid future target:
# is_declining_label is derived from trend_direction, which is derived from trend_pct.
label_columns = [c for c in ["is_declining_label", "trend_direction", "trend_pct"] if c in df.columns]
print("Starter decline-related columns present:", label_columns)

target_name = "future_decline_target"
print(f"Planned target column: {target_name}")
print("Definition: observed decline in a later evaluation window after the snapshot.")


Starter decline-related columns present: ['trend_direction', 'trend_pct']
Planned target column: future_decline_target
Definition: observed decline in a later evaluation window after the snapshot.


## 3. Success metric

**Primary success metric: Precision@50.**

A useful model should put genuinely declining pages near the top of the review queue. Precision@50 answers: *of the 50 pages we ask editors to review first, how many actually show the observed future decline outcome?*

This metric matches the real capacity constraint: editors have a finite review budget, so the quality of the top of the ranked queue matters more than performance on every page.

In [5]:
# Precision@K is defined for a ranked queue. For this framing notebook we name K explicitly.
K = 50
print(f"Primary metric: Precision@{K}")
print("Interpretation: among the top 50 ranked pages, the share with the observed future decline outcome.")


Primary metric: Precision@50
Interpretation: among the top 50 ranked pages, the share with the observed future decline outcome.


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one row = one pseudonymized content item/page at the snapshot.**

The starter dataset contains 30,000 rows and is described as one row per pseudonymized content item, with trailing-90-day performance metrics. I will use a small lane slice for inspection and keep `content_id` / `client_id` only for identification and grouping, not as model features.

The code below discovers useful columns from the actual starter CSV instead of hard-coding a column list that could drift. It also creates a `future_decline_target` placeholder with missing values, because the future observed outcome is not available in this Week-2 starter snapshot.

In [6]:
# Show a real dataframe for the unit of analysis.
# Keep identifiers for inspection/grouping, but do not use them as model features.
preferred = [
    "content_id", "client_id", "content_type", "age_days",
    "word_count", "clicks_90d", "impressions_90d", "ctr",
    "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "search_volume"
]
available = [c for c in preferred if c in df.columns]

# If a preferred metric is absent, add other non-ID columns so the dataframe is still useful.
if len(available) < 4:
    fallback = [c for c in df.columns if c not in {"content_id", "client_id"}]
    available = (["content_id", "client_id"] if "content_id" in df.columns else []) + fallback[:8]

lane_df = df[available].head(10).copy()
lane_df["future_decline_target"] = pd.NA

print("One row = one pseudonymized content item/page at the snapshot.")
print("Displayed columns:", available + ["future_decline_target"])
display(lane_df)

print("\nShape of full starter data:", df.shape)


One row = one pseudonymized content item/page at the snapshot.
Displayed columns: ['content_id', 'client_id', 'content_type', 'word_count', 'clicks_90d', 'impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'search_volume', 'future_decline_target']


,content_id,client_id,content_type,word_count,clicks_90d,impressions_90d,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,search_volume,future_decline_target
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,29,3803,0.76,10.6,5.88,4.55,0.0,10.0,<NA>
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,7,15320,0.05,20.3,0.00,10.00,0.0,90.0,<NA>
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,11,12581,0.09,36.5,0.00,28.57,0.0,0.0,<NA>
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,58,11751,0.49,6.2,1.28,3.45,0.0,10.0,<NA>
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,24,19140,0.13,44.0,0.00,24.29,0.0,0.0,<NA>
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3080.0,1,3970,0.03,8.5,0.00,25.00,0.0,720.0,<NA>
6,content_9a34b442b552,client_8722616204,keyword article,3059.0,0,20,0.00,7.0,0.00,0.00,0.0,0.0,<NA>
7,content_a63219c6e95a,client_19581e27de,keyword article,NaN,1,1724,0.06,21.2,3.57,7.14,0.0,590.0,<NA>
8,content_5e6c160719bc,client_6208ef0f77,keyword article,3807.0,29,32574,0.09,46.0,5.88,6.25,0.0,0.0,<NA>
9,content_c27558df2b0c,client_19581e27de,keyword article,NaN,2,1240,0.16,4.9,0.00,0.00,0.0,0.0,<NA>



Shape of full starter data: (30000, 44)


## 5. Why ML beats a fixed rule here

A fixed rule is a useful baseline, but it becomes brittle when many signals interact.

For example, a page may deserve attention because of some combination of recent traffic, clicks, CTR, search position, engagement, content age, content type, and missingness/data availability. The useful pattern may also differ by client and change over time. Writing and maintaining every interaction as nested `if` statements would be difficult to audit and update.

ML earns its place if it can learn these multi-signal patterns from **observed future outcomes** and then improve the top of the editorial queue on held-out data. The fixed rule remains important as a transparent baseline; the ML system must beat it on the pre-declared Precision@50 metric before it is useful.

**Claim level:** decision-support based on observed/measured outcomes. This is not a causal claim and not a prediction of Google's ranking algorithm.

In [7]:
# Simple framing checks: IDs are not features, and leakage columns are explicitly excluded.
id_cols = [c for c in ["content_id", "client_id"] if c in df.columns]
leakage_cols = [c for c in ["is_declining_label", "trend_direction", "trend_pct"] if c in df.columns]

print("Identification/grouping columns (not model features):", id_cols)
print("Leakage/derived decline columns (not model features):", leakage_cols)
print("Model target is planned as a later observed outcome:", target_name)


Identification/grouping columns (not model features): ['content_id', 'client_id']
Leakage/derived decline columns (not model features): ['trend_direction', 'trend_pct']
Model target is planned as a later observed outcome: future_decline_target


## Self-check

- [x] Task type, target/proxy, and success metric are explicitly named.
- [x] The decision, actor, and cost of a wrong call are stated.
- [x] The unit of analysis is shown as a real dataframe by the code cell.
- [x] The future target is framed as an observed later outcome, not a label invented from the current rule.
- [x] `trend_direction` and `trend_pct` are not used as model features because they define the starter decline label.
- [x] IDs are treated as grouping/identification fields, not model features.
- [x] Claims are limited to observed/measured/directional decision-support language.
- [ ] Run **Runtime → Run all** after the starter CSV is available at `data/raw/content_refresh_anonymized.csv`, then save/commit the executed notebook under `work/notebooks/w02_ml_task_framing.ipynb`.
